# Task 3 — Real-World Dataset Analysis Project
**Candidate:** Ganesh  
**Internship:** YuvaIntern Data Analyst Internship

## Project: Online Retail Transaction Analysis
**Dataset:** UCI Machine Learning Repository — Online Retail (Chen, 2015), DOI: https://doi.org/10.24432/C5BW33

**Important:** This notebook fetches the original UCI dataset when run. All numerical findings and charts are calculated from the downloaded real data. No synthetic records or invented results are used.


## 1. Business context, objectives and questions
The UCI Online Retail dataset contains transactions from a UK-based non-store online retailer between 1 December 2010 and 9 December 2011. The business sells giftware and has many wholesale customers.

Questions:
1. How does sales revenue change over time?
2. Which products and countries contribute most to revenue?
3. How concentrated is revenue among customers?
4. What data-quality issues must be addressed?
5. What operational recommendations are supported by the analysis?

**Metric definition:** line revenue = Quantity × UnitPrice, in GBP. Cancellation invoices and non-positive quantity/price lines are excluded from the main positive-sales analysis, and are counted separately as data-quality/business events. This definition is explicit and should be retained when comparing results.


## 2. Environment setup and data acquisition
Run this cell in Google Colab or Jupyter. The UCI package downloads the dataset directly from the repository.

In [ ]:
%pip -q install ucimlrepo openpyxl seaborn

import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

online_retail = fetch_ucirepo(id=352)
raw = online_retail.data.features.copy()
print("Downloaded shape:", raw.shape)
display(raw.head())


## 3. Dataset source, variables and provenance

In [ ]:
print("Dataset metadata:")
print(online_retail.metadata)
print("\nVariable definitions:")
display(online_retail.variables)
print("\nColumns:", raw.columns.tolist())
print("\nData types:")
print(raw.dtypes)


## 4. Initial exploration and quality audit

In [ ]:
audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean()*100).round(2),
    "unique_values": raw.nunique(dropna=True)
})
display(audit)
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Cancellation invoice rows:", int(raw["InvoiceNo"].astype(str).str.startswith("C", na=False).sum()))
print("Non-positive quantity rows:", int((pd.to_numeric(raw["Quantity"], errors="coerce") <= 0).sum()))
print("Non-positive unit price rows:", int((pd.to_numeric(raw["UnitPrice"], errors="coerce") <= 0).sum()))
display(raw.describe(include="all").T)


## 5. Cleaning and preprocessing
Decisions:
- Preserve `raw`; create a separate working copy.
- Remove exact duplicate rows.
- Parse invoice dates with coercion and inspect parsing failures.
- Convert invoice and stock codes to strings to preserve identifier semantics.
- Exclude cancellation invoices (invoice number begins with `C`) from positive sales calculations.
- Exclude rows with non-positive quantity or unit price from the main revenue analysis. These may represent returns, cancellations, or adjustments and are counted rather than silently treated as ordinary sales.
- Do not impute missing CustomerID: customer-level segmentation requires a genuine identifier. Retain such rows in product/country/time analyses where valid.
- Remove rows missing essential product description, date, quantity, or price for the relevant line-sales analysis.
- Flag extreme values using the IQR rule for investigation; do not automatically delete legitimate wholesale orders.


In [ ]:
data = raw.copy()
data = data.drop_duplicates().copy()

data["InvoiceNo"] = data["InvoiceNo"].astype("string")
data["StockCode"] = data["StockCode"].astype("string")
data["Description"] = data["Description"].astype("string").str.strip()
data["Country"] = data["Country"].astype("string").str.strip()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], errors="coerce")
for col in ["Quantity", "UnitPrice", "CustomerID"]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

cancel_mask = data["InvoiceNo"].str.startswith("C", na=False)
nonpositive_qty = data["Quantity"].le(0)
nonpositive_price = data["UnitPrice"].le(0)
print("Cancellation rows:", int(cancel_mask.sum()))
print("Rows with quantity <= 0:", int(nonpositive_qty.sum()))
print("Rows with unit price <= 0:", int(nonpositive_price.sum()))

sales = data.loc[
    (~cancel_mask) &
    data["InvoiceDate"].notna() &
    data["Description"].notna() &
    data["Quantity"].gt(0) &
    data["UnitPrice"].gt(0)
].copy()

sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["InvoiceMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)
sales["InvoiceDay"] = sales["InvoiceDate"].dt.date

print("Rows after cleaning:", len(sales))
print("Remaining missing values in key analysis columns:")
display(sales[["InvoiceDate","Description","Quantity","UnitPrice","Revenue","Country"]].isna().sum())
print("CustomerID missing %:", round(sales["CustomerID"].isna().mean()*100,2))


## 6. Outlier investigation
The IQR rule flags unusually high line revenue and quantity. This is a screening tool, not proof of error: wholesale purchases can be genuinely large. Compare results with and without flagged rows and document the effect.

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

qty_lo, qty_hi = iqr_bounds(sales["Quantity"])
rev_lo, rev_hi = iqr_bounds(sales["Revenue"])
sales["Quantity_IQR_Flag"] = (sales["Quantity"] < qty_lo) | (sales["Quantity"] > qty_hi)
sales["Revenue_IQR_Flag"] = (sales["Revenue"] < rev_lo) | (sales["Revenue"] > rev_hi)
print("Quantity IQR bounds:", qty_lo, qty_hi)
print("Revenue IQR bounds:", rev_lo, rev_hi)
print("Quantity flagged:", int(sales["Quantity_IQR_Flag"].sum()))
print("Revenue flagged:", int(sales["Revenue_IQR_Flag"].sum()))


## 7. Exploratory data analysis — key metrics

In [ ]:
total_revenue = sales["Revenue"].sum()
unique_invoices = sales["InvoiceNo"].nunique()
unique_products = sales["StockCode"].nunique()
unique_customers = sales["CustomerID"].nunique()
avg_line_revenue = sales["Revenue"].mean()

kpis = pd.Series({
    "Positive-sales line revenue (GBP)": total_revenue,
    "Invoice count": unique_invoices,
    "Distinct products": unique_products,
    "Identified customers": unique_customers,
    "Average line revenue (GBP)": avg_line_revenue
})
display(kpis.to_frame("Value").round(2))


## 8. Time trend analysis

In [ ]:
monthly = sales.groupby("InvoiceMonth").agg(
    Revenue=("Revenue","sum"),
    Invoices=("InvoiceNo","nunique"),
    Units=("Quantity","sum")
).sort_index()
display(monthly.round(2))

ax = monthly["Revenue"].plot(figsize=(12,5), marker="o", title="Monthly Revenue Trend")
ax.set_xlabel("Invoice month"); ax.set_ylabel("Revenue (GBP)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


## 9. Product and country analysis

In [ ]:
product_rank = sales.groupby("Description").agg(
    Revenue=("Revenue","sum"),
    Units=("Quantity","sum"),
    Invoices=("InvoiceNo","nunique")
).sort_values("Revenue", ascending=False)
country_rank = sales.groupby("Country").agg(
    Revenue=("Revenue","sum"),
    Invoices=("InvoiceNo","nunique"),
    Customers=("CustomerID","nunique")
).sort_values("Revenue", ascending=False)

print("Top 15 products by revenue:")
display(product_rank.head(15).round(2))
print("Top 15 countries by revenue:")
display(country_rank.head(15).round(2))

product_rank.head(10)["Revenue"].sort_values().plot(kind="barh", figsize=(9,6), title="Top 10 Products by Revenue")
plt.xlabel("Revenue (GBP)"); plt.tight_layout(); plt.show()

country_rank.head(10)["Revenue"].sort_values().plot(kind="barh", figsize=(9,6), title="Top 10 Countries by Revenue")
plt.xlabel("Revenue (GBP)"); plt.tight_layout(); plt.show()


## 10. Customer analysis and RFM segmentation
RFM uses Recency (days since last purchase relative to the dataset's last date), Frequency (distinct invoices), and Monetary (revenue). Customers without a CustomerID are excluded only from customer-level analysis. Quartile scores are descriptive segments, not validated predictions.

In [ ]:
identified = sales.dropna(subset=["CustomerID"]).copy()
identified["CustomerID"] = identified["CustomerID"].astype("int64")
snapshot_date = identified["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = identified.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo","nunique"),
    Monetary=("Revenue","sum")
)
rfm["R_score"] = pd.qcut(rfm["Recency"].rank(method="first"), 4, labels=[4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["RFM_score"] = rfm[["R_score","F_score","M_score"]].astype(str).agg("".join, axis=1)
display(rfm.sort_values("Monetary", ascending=False).head(10))
print("RFM customer count:", len(rfm))


## 11. Distribution, relationships, and visualization

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,4))
sns.histplot(sales["Quantity"].clip(upper=sales["Quantity"].quantile(.99)), bins=40, ax=axes[0])
axes[0].set_title("Quantity Distribution (clipped at 99th percentile)")
sns.histplot(sales["Revenue"].clip(upper=sales["Revenue"].quantile(.99)), bins=40, ax=axes[1])
axes[1].set_title("Line Revenue Distribution (clipped at 99th percentile)")
plt.tight_layout(); plt.show()

sample = sales.sample(min(10000, len(sales)), random_state=42)
plt.figure(figsize=(8,5))
sns.scatterplot(data=sample, x="Quantity", y="Revenue", alpha=.35)
plt.title("Quantity vs Line Revenue (sample for readability)")
plt.tight_layout(); plt.show()

corr = sales[["Quantity","UnitPrice","Revenue"]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()
display(corr.round(3))


## 12. Findings generated from the data

In [ ]:
print(f"Total positive-sales revenue: GBP {total_revenue:,.2f}")
print("Top revenue country:", country_rank.index[0], f"(GBP {country_rank.iloc[0]['Revenue']:,.2f})")
print("Top revenue product:", product_rank.index[0], f"(GBP {product_rank.iloc[0]['Revenue']:,.2f})")
print("Peak revenue month:", monthly["Revenue"].idxmax(), f"(GBP {monthly['Revenue'].max():,.2f})")
print("Revenue share of top 10 products:",
      round(product_rank.head(10)["Revenue"].sum()/total_revenue*100,2), "%")
print("Customer-level identified revenue share:",
      round(identified["Revenue"].sum()/total_revenue*100,2), "%")


## 13. Business interpretation and recommendations
Use the actual output above to write conclusions. Do not copy generic claims without checking the computed tables and charts.

Recommended decision framework:
- **Merchandising:** prioritize inventory review for products with high revenue and/or high unit volume; check margin before calling them most profitable.
- **Geographic strategy:** compare country revenue with order count and customer count; high revenue alone does not prove high profitability.
- **Customer retention:** use RFM groups to identify high-value recent customers and customers whose recency is worsening; validate segments over time.
- **Operations:** investigate cancellation and return patterns separately rather than mixing negative quantities into ordinary sales.
- **Data quality:** track missing customer identifiers because they limit customer-level attribution.


## 14. Limitations
- The dataset covers one retailer and a historical period (2010–2011); it is not representative of today's entire retail market.
- Revenue is not profit: costs, shipping, taxes, and margins are not supplied.
- Customer IDs are missing for some records, limiting customer-level conclusions.
- Large orders may be legitimate wholesale purchases; IQR flags require domain review.
- Correlation does not establish causation.
- RFM scores depend on the chosen snapshot date and quartile method.


## 15. Reproducibility and references
**Dataset citation:** Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. Licensed CC BY 4.0.

Run all cells from top to bottom. The notebook downloads the dataset through `ucimlrepo`; record the execution date and library versions when submitting. Export the executed notebook with outputs visible, and include the generated charts and summary tables in the final report.
